In [ ]:
<!DOCTYPE html>
<html lang="zh-CN">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>掌心绽放的照片</title>
    <script src="https://cdn.jsdelivr.net/npm/@mediapipe/camera_utils/camera_utils.js" crossorigin="anonymous"></script>
    <script src="https://cdn.jsdelivr.net/npm/@mediapipe/control_utils/control_utils.js" crossorigin="anonymous"></script>
    <script src="https://cdn.jsdelivr.net/npm/@mediapipe/drawing_utils/drawing_utils.js" crossorigin="anonymous"></script>
    <script src="https://cdn.jsdelivr.net/npm/@mediapipe/hands/hands.js" crossorigin="anonymous"></script>

    <style>
        body {
            display: flex;
            justify-content: center;
            align-items: center;
            height: 100vh;
            background-color: #1a1a1a;
            margin: 0;
            overflow: hidden;
            font-family: sans-serif;
        }

        /* 隐藏摄像头视频，我们只需要它的数据 */
        .input_video {
            display: none;
        }

        /* 容器 */
        .stage {
            position: relative;
            width: 300px;
            height: 300px;
            display: flex;
            justify-content: center;
            align-items: center;
        }

        /* 核心元素：照片 */
        .photo-card {
            width: 100%;
            height: 100%;
            /* 替换这里为你自己的照片地址 */
            background-image: url('https://images.unsplash.com/photo-1518020382113-a7e8fc38eac9?ixlib=rb-1.2.1&auto=format&fit=crop&w=800&q=80');
            background-size: cover;
            background-position: center;

            /* 关键动画属性 */
            transition: all 0.8s cubic-bezier(0.68, -0.55, 0.27, 1.55);

            /* 默认状态：爱心形状 */
            clip-path: path("M150 270 C 150 270 20 180 20 90 A 65 65 0 0 1 150 90 A 65 65 0 0 1 280 90 C 280 180 150 270 150 270 Z");
            background-color: #ff4d6d; /* 照片加载前显示的粉色 */
            transform: scale(0.8);
            filter: brightness(0.9);
            box-shadow: 0 0 50px rgba(255, 77, 109, 0.6);
        }

        /* 绽放状态：当添加了 .bloom 类名 */
        .photo-card.bloom {
            /* 变成矩形 (inset(0) 代表没有裁剪) */
            clip-path: inset(0% 0% 0% 0% round 15px);
            transform: scale(1.1);
            filter: brightness(1.1);
            box-shadow: 0 20px 60px rgba(0,0,0,0.5);
        }

        /* 加载提示 */
        .loading {
            position: absolute;
            color: white;
            top: 20px;
            left: 20px;
            z-index: 10;
        }

        /* 调试用的手势点（可选） */
        .canvas-output {
            position: absolute;
            bottom: 10px;
            right: 10px;
            width: 160px;
            height: 120px;
            border: 1px solid rgba(255,255,255,0.3);
            opacity: 0.5;
            transform: scaleX(-1); /* 镜像翻转 */
        }
    </style>
</head>
<body>

    <div class="loading">正在启动摄像头识别...<br>请在摄像头前张开手掌</div>

    <video class="input_video"></video>

    <div class="stage">
        <div class="photo-card" id="myPhoto"></div>
    </div>

    <canvas class="canvas-output"></canvas>

    <script>
        const videoElement = document.getElementsByClassName('input_video')[0];
        const canvasElement = document.getElementsByClassName('canvas-output')[0];
        const canvasCtx = canvasElement.getContext('2d');
        const photoCard = document.getElementById('myPhoto');
        const loadingText = document.querySelector('.loading');

        // 手势处理函数
        function onResults(results) {
            // 绘制调试画面
            canvasCtx.save();
            canvasCtx.clearRect(0, 0, canvasElement.width, canvasElement.height);
            canvasCtx.drawImage(results.image, 0, 0, canvasElement.width, canvasElement.height);

            if (results.multiHandLandmarks && results.multiHandLandmarks.length > 0) {
                // 隐藏加载文字
                loadingText.style.display = 'none';

                // 获取第一只手的数据
                const landmarks = results.multiHandLandmarks[0];
                drawConnectors(canvasCtx, landmarks, HAND_CONNECTIONS, {color: '#00FF00', lineWidth: 1});

                // 核心逻辑：判断手是张开的还是握住的
                // 我们计算 大拇指指尖(4) 和 小拇指指尖(20) 的距离
                const thumbTip = landmarks[4];
                const pinkyTip = landmarks[20];

                // 计算两点间距离 (勾股定理)
                const distance = Math.sqrt(
                    Math.pow(thumbTip.x - pinkyTip.x, 2) +
                    Math.pow(thumbTip.y - pinkyTip.y, 2)
                );

                // 距离阈值：根据你的摄像头距离可能需要微调
                // 0.3 是一个经验值，表示两指跨度占屏幕宽度的比例
                // 大于 0.35 认为是张手，小于 0.25 认为是缩手 (设置缓冲区防止闪烁)
                if (distance > 0.30) {
                    photoCard.classList.add('bloom'); // 绽放
                } else if (distance < 0.20) {
                    photoCard.classList.remove('bloom'); // 回到爱心
                }
            }
            canvasCtx.restore();
        }

        // 初始化 MediaPipe Hands
        const hands = new Hands({locateFile: (file) => {
            return `https://cdn.jsdelivr.net/npm/@mediapipe/hands/${file}`;
        }});

        hands.setOptions({
            maxNumHands: 1, // 只识别一只手
            modelComplexity: 1,
            minDetectionConfidence: 0.5,
            minTrackingConfidence: 0.5
        });

        hands.onResults(onResults);

        // 启动摄像头
        const camera = new Camera(videoElement, {
            onFrame: async () => {
                await hands.send({image: videoElement});
            },
            width: 640,
            height: 480
        });
        camera.start();
    </script>
</body>
</html>